# Spanish Transcritption

## Installations

In [1]:
!pip install python-Levenshtein

In [2]:
!pip install pyannote.audio==3.1.1 --upgrade

In [3]:
!pip install faster_whisper

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached ctranslate2-4.7.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
  Using cached av-17.0.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (4.6 kB)
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
Using cached av-17.0.0-cp311-abi3-manylinux_2_28_x86_64.whl (36.4 MB)
Using cached ctranslate2-4.7.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (39.0 MB)
Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (17.2 MB)


In [4]:
!pip install pyannote.audio


In [5]:
!pip install noisereduce

In [6]:
!pip install python-Levenshtein openpyxl -q

## Imports


In [7]:
import numpy as np
import torch
import noisereduce as nr
import soundfile as sf
from faster_whisper import WhisperModel
from pydub import AudioSegment
import tempfile, os
import Levenshtein
import re
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Preprocesssing

### Load and Resample

This function is responsible for converting raw audio input into a format suitable for speech recognition.

The input audio file may come in various formats (MP3, WAV, etc.), sampling rates, and channel configurations. However, the transcription model expects a very specific format. This function standardizes the audio to meet those requirements. The audio is first loaded using pydub. Next, the sampling rate is resampled to 16 kHz, which is the expected input for Whisper-based models.

The waveform is then converted into a NumPy array of floating-point values and normalized to the range [-1, 1]. This step ensures standardized signals.

In [9]:
def load_and_resample(path: str, target_sr: int = 16000):
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max
    return samples, target_sr

### DC Offset

This function removes any constant bias (DC offset) present in the audio signal.

In some recordings, the waveform may be slightly shifted above or below zero, meaning it is not centered around zero amplitude. This offset does not carry useful information and can affect signal processing operations.

By subtracting the mean of the signal, the waveform is re-centered around zero. This ensures a balanced signal and improves the effectiveness of subsequent processing steps

In [10]:
def remove_dc_offset(samples):
    return samples - np.mean(samples)

### VAD

This function identifies regions of the audio that contain speech and filters out silence or non-speech segments to isolate speech in signal. This is done to improve computational efficieny and signal quality.

The implementation uses the Silero VAD model, loaded via torch.hub. This model is trained to detect speech boundaries in raw audio signals and outputs timestamps corresponding to speech segments.

The input waveform is first converted into a PyTorch tensor, as required by the model. The VAD then processes the signal and returns a list of timestamps indicating where speech begins and ends.

Several parameters control the behavior of the detector:

`threshold`: Controls sensitivity. Lower values detect more speech (including noise), while higher values are stricter.

`min_speech_duration_ms`: Filters out very short segments that are unlikely to contain meaningful speech.

`min_silence_duration_ms`: Determines how much silence is required to separate two speech segments.

The output is converted from sample indices to time (in seconds), producing a list of (start_time, end_time) tuples representing detected speech regions.

In [11]:
def run_vad(samples: np.ndarray, sr: int, thr: float):
    model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False
    )
    (get_speech_timestamps, _, read_audio, *_) = utils

    audio_tensor = torch.FloatTensor(samples)
    speech_timestamps = get_speech_timestamps(
        audio_tensor, model,
        sampling_rate=sr,
        threshold=thr,
        min_speech_duration_ms=400,
        min_silence_duration_ms=500
    )
    segments = [(t["start"] / sr, t["end"] / sr) for t in speech_timestamps]
    print(f"[VAD] Found {len(segments)} speech segments")
    return segments

### Add Padding

This step expands each speech segment slightly by adding a small amount of time before and after the detected boundaries.

Voice Activity Detection can sometimes cut too tightly, trimming parts of words at the beginning or end of a segment. Padding helps recover this lost context by extending the segment boundaries.

This ensures that when the audio is later sliced for transcription, the model receives more complete speech, leading to better sentence reconstruction and fewer fragmented outputs.

In [12]:
def add_padding(segments, sr, pad_ms=200):
    padded = []
    pad = int(pad_ms * sr / 1000)

    for s, e in segments:
        new_s = max(0, s - pad/sr)
        new_e = e + pad/sr
        padded.append((new_s, new_e))

    return padded

### Concatenate only the VAD speech chunks





This function extracts and concatenates only speech regions into a single continuous audio stream.

 **Note: It is not used in the final pipeline, as it removes temporal boundaries required for sentence-level alignment and matching. It was explored in earlier iterations but discarded in favor of segment-wise transcription.**

In [13]:
def extract_speech_audio(samples: np.ndarray, sr: int, segments):
    chunks = [samples[int(s * sr): int(e * sr)] for s, e in segments]
    speech_audio = np.concatenate(chunks) if chunks else samples
    print(f"[VAD] Kept {len(speech_audio)/sr:.1f}s of speech "
          f"from {len(samples)/sr:.1f}s total")
    return speech_audio

### Merge Vad Segments before

This function merges nearby speech segments that are separated by very short gaps.

Voice Activity Detection often produces fragmented outputs, where a single sentence may be split into multiple smaller segments due to brief pauses or minor silence within speech. These fragments can lead to incomplete or disjointed transcriptions.

To address this, segments are merged if the gap between them is less than a specified threshold `(max_gap)=0.5`. This helps reconstruct longer, more coherent speech regions that better align with full sentences.

In [14]:
def merge_vad_segments(segments, max_gap=0.5):
    merged = []
    current_start, current_end = segments[0]

    for s, e in segments[1:]:
        if s - current_end <= max_gap:
            current_end = e
        else:
            merged.append((current_start, current_end))
            current_start, current_end = s, e

    merged.append((current_start, current_end))
    return merged

### Noise Reduction

This function applies spectral noise reduction to the audio signal to suppress background noise and improve speech clarity.

It uses the noisereduce library, which estimates noise characteristics from the signal and reduces unwanted components while preserving speech. The parameter prop_decrease controls the strength of noise suppression.

Noise reduction can help improve transcription quality in recordings with significant background noise. However, excessive noise reduction may distort speech and degrade model performance.

For this reason, this step is kept optional in the pipeline and can be enabled or disabled depending on the audio quality.

**Performance with and without Noise Reduction has been tested ahead**

> **Note:** Applying spectral noise reduction (`reduce_noise`) did not significantly change the evaluation statistics in tests.

In [15]:
def reduce_noise(samples: np.ndarray, sr: int, pd: float):
    denoised = nr.reduce_noise(y=samples, sr=sr, stationary=False, prop_decrease=pd)
    print("[Noise Reduction] Done")
    return denoised

## Transcription

### Transcribe

This function converts the speech segments into text using a Whisper-based speech recognition model.

For each detected speech segment, the corresponding portion of the audio is extracted and passed to the transcription model. Very short segments (less than 0.5 seconds) are skipped, as they are unlikely to contain meaningful speech and may introduce noise.

The model processes each segment independently and returns transcribed text along with timing information. Since transcription is performed on sliced segments, the timestamps are adjusted (offset) to align with the original audio timeline.

The outputs are stored as a list of dictionaries, each containing:

`start time`

`end time`

`transcribed text`

`detected language`

This segment-wise transcription approach preserves temporal structure and allows for fine-grained alignment with target sentences in later stages.

In [16]:
def transcribe(samples: np.ndarray, sr: int, speech_segments: list):
    model = WhisperModel("medium", device="cuda", compute_type="float16")

    results = []
    for i, (start, end) in enumerate(speech_segments):
        chunk = samples[int(start * sr): int(end * sr)]

        # Skip very short chunks
        if len(chunk) / sr < 0.5:
            continue

        segments, info = model.transcribe(
            chunk,
            language= "es",
            beam_size=5,
            vad_filter=False,
        )

        for seg in segments:
            results.append({
                "start": start + seg.start,
                "end":   start + seg.end,
                "text":  seg.text.strip(),
                "lang":  info.language,
            })
            print(f"  [{start + seg.start:.1f}s → start + {seg.end:.1f}s] "
                  f"[{info.language}] {seg.text.strip()}")

    return results

### Merging

**NOTE: not used in pipeline**

A bruteforce method to create 30 segments of text that did not work at all

## Post Processing

In [17]:
def merge_until_target(results, target_count=30):
    current = results.copy()

    while len(current) > target_count:
        lengths = [len(r["text"].split()) for r in current]
        idx = lengths.index(min(lengths))

        if idx == len(current) - 1:
            merge_idx = idx - 1
        else:
            merge_idx = idx

        merged = {
            "start": current[merge_idx]["start"],
            "end": current[merge_idx + 1]["end"],
            "text": current[merge_idx]["text"] + " " + current[merge_idx + 1]["text"]
        }

        current = (
            current[:merge_idx] +
            [merged] +
            current[merge_idx + 2:]
        )

    return current

### Stimulus List

In [18]:
STIMULI = [
    (1,  "Quiero cortarme el pelo"),
    (2,  "El libro está en la mesa"),
    (3,  "El carro lo tiene Pedro"),
    (4,  "El se ducha cada mañana"),
    (5,  "¿Qué dice usted que va a hacer hoy?"),
    (6,  "Dudo que sepa manejar muy bien"),
    (7,  "Las calles de esta ciudad son muy anchas"),
    (8,  "Puede que llueva mañana todo el día"),
    (9,  "Las casas son muy bonitas pero caras"),
    (10, "Me gustan las películas que acaban bien"),
    (11, "El chico con el que yo salgo es español"),
    (12, "Después de cenar me fui a dormir tranquilo"),
    (13, "Quiero una casa en la que vivan mis animales"),
    (14, "A nosotros nos fascinan las fiestas grandiosas"),
    (15, "Ella sólo bebe cerveza y no come nada"),
    (16, "Me gustaría que el precio de las casas bajara"),
    (17, "Cruza a la derecha y después sigue todo recto"),
    (18, "Ella ha terminado de pintar su apartamento"),
    (19, "Me gustaría que empezara a hacer más calor pronto"),
    (20, "El niño al que se le murió el gato está triste"),
    (21, "Una amiga mía cuida a los niños de mi vecino"),
    (22, "El gato que era negro fue perseguido por el perro"),
    (23, "Antes de poder salir él tiene que limpiar su cuarto"),
    (24, "La cantidad de personas que fuman ha disminuido"),
    (25, "Después de llegar a casa del trabajo tomé la cena"),
    (26, "El ladrón al que atrapó la policía era famoso"),
    (27, "Le pedí a un amigo que me ayudara con la tarea"),
    (28, "El examen no fue tan difícil como me habían dicho"),
    (29, "¿Serías tan amable de darme el libro que está en la mesa?"),
    (30, "Hay mucha gente que no toma nada para el desayuno"),
]


# ── Drop this into your pipeline after transcribe() ──────────────────────────


### Merging Nearby fragments

This step merges consecutive transcription segments that are very close in time. If the gap between two segments is below a threshold (e.g., 2 seconds), they are likely part of the same sentence but were split due to pauses or VAD segmentation.

Instead of keeping them separate, they are merged into a single segment, and a special <pause> token is inserted to preserve the pause information.

In [19]:
## merging 2 transcriptions if they have a gap of less than 2 seconds, and adding a <pause> token.
def merge_nearby_fragments(results, gap_threshold_s=2.0):
    if not results:
        return results

    merged = [results[0].copy()]
    for curr in results[1:]:
        prev = merged[-1]
        gap = curr["start"] - prev["end"]
        if gap <= gap_threshold_s:
            prev["end"]  = curr["end"]
            prev["text"] = prev["text"].rstrip() + " <pause> " + curr["text"].lstrip()  # ← only change
        else:
            merged.append(curr.copy())

    print(f"[Merge] {len(results)} segments → {len(merged)} after merging (gap ≤ {gap_threshold_s}s)")
    return merged




### Normalizing

Removing `<pause>` token to show when 2 outputs have been merged. The `<pause>` token indicates that the speaker also took a longer pause, thereby indicating disfluency

In [20]:
def normalize(text):
    text = text.replace("<pause>", "")
    return re.sub(r'[^\w\s]', '', text.lower().strip())

### Candidate Window Function


This function constructs candidate text segments by combining consecutive ASR outputs into larger windows. Instead of relying on individual transcription fragments, it generates sequences of length 1 up to a specified maximum window size (e.g., 3).

For each starting segment, it progressively concatenates the next segments, creating candidates such as:

- single segment
- two consecutive segments
- three consecutive segments

Each candidate retains:
- combined text
- start timestamp of the first segment
- end timestamp of the last segment

This allows the system to recover full sentences that were split across multiple ASR fragments due to pauses or VAD segmentation.

For example:

"el chico que" + "hago es español" → "el chico que hago es español"

Without this step, matching would fail for fragmented sentences.

In [21]:
def build_candidate_windows(results, max_window_size=3):
    candidates = []

    for i in range(len(results)):
        text = results[i]["text"]
        start = results[i]["start"]
        end = results[i]["end"]

        # window size 1, 2, 3
        for j in range(1, max_window_size + 1):
            if i + j > len(results):
                break

            if j > 1:
                next_seg = results[i + j - 1]
                text += " " + next_seg["text"]
                end = next_seg["end"]

            candidates.append({
                "start": start,
                "end": end,
                "text": text.strip()
            })

    print(f"[Windowing] Generated {len(candidates)} candidates")
    return candidates

### Matching ASR to Stimuli

This function aligns transcribed audio segments with predefined stimulus sentences using a window-based matching strategy.

Instead of matching individual ASR fragments directly, it first generates candidate windows by combining consecutive segments. This allows the system to compare full or partial sentences rather than fragmented text.

For each stimulus sentence:

- all candidate windows are evaluated
- the Levenshtein distance is computed between normalized candidate text and the stimulus
- the candidate with the lowest distance is selected
    
To ensure consistent alignment, the function prevents reuse of overlapping audio regions. Once a candidate is assigned to a stimulus, its time span is marked as used and excluded from future matches.

A threshold (`max_dist`) is applied to control match quality:

- if the best match is within the threshold → it is accepted
- otherwise → the sentence is marked as `[not produced]`
    
---

**Greedy Approximation Strategy**

This matching process follows a greedy approximation approach. Instead of solving the global optimal alignment problem (which would require expensive dynamic programming or combinatorial search), the algorithm makes locally optimal decisions for each stimulus.

At each step:

- the best available candidate (with minimum edit distance) is selected
- overlapping regions are removed from future consideration
    
This simplifies the problem significantly while remaining effective in practice, because:

- speech follows a sequential structure
- sentences occur in order
- most transcription errors are local rather than global
    
As a result, the algorithm achieves a good balance between:

- computational efficiency
- alignment accuracy
    
---

**Distance Metric**

The distance metric used is Levenshtein distance, which measures the minimum number of single-character edits (insertions, deletions, or substitutions) required to transform one string into another. This makes it robust to typical ASR errors such as:

- missing words
- incorrect words
- minor spelling variations


In [22]:
def match_asr_to_stimuli(results, stimuli, max_dist=20):
    candidates = build_candidate_windows(results)

    used_spans = []
    matched = []

    for num, stim in stimuli:
        best_score = float('inf')
        best_text = "[not produced]"
        best_span = None

        for c in candidates:
            # avoid overlapping already-used segments
            overlap = any(not (c["end"] <= u[0] or c["start"] >= u[1]) for u in used_spans)
            if overlap:
                continue

            dist = Levenshtein.distance(
                normalize(c["text"]),
                normalize(stim)
            )

            if dist < best_score:
                best_score = dist
                best_text = c["text"]
                best_span = (c["start"], c["end"])

        if best_score <= max_dist:
            used_spans.append(best_span)
            transcription = best_text
        else:
            transcription = "[not produced]"

        matched.append({
            "sentence": num,
            "stimulus": stim,
            "transcription": transcription,
        })

        print(f"{num:>2}. [{stim}]")
        print(f"    → {transcription}  (dist={best_score})\n")

    return matched

### Export

Exports it into an Excel Sheet

In [23]:
### export
def export_to_excel(matched, output_path):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Transcriptions"

    thin = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Header
    for col, label in enumerate(["Sentence", "Stimulus", "Transcription"], 1):
        cell = ws.cell(row=1, column=col, value=label)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        cell.fill      = PatternFill("solid", start_color="4472C4")
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border

    # Data rows
    for r in matched:
        ws.append([r["sentence"], r["stimulus"], r["transcription"]])
        for col in range(1, 4):
            cell = ws.cell(row=ws.max_row, column=col)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            cell.border    = border
            cell.font      = Font(name="Arial", size=11)

    ws.column_dimensions["A"].width = 10
    ws.column_dimensions["B"].width = 52
    ws.column_dimensions["C"].width = 52
    ws.row_dimensions[1].height     = 20

    wb.save(output_path)
    print(f"\n[Export] Saved → {output_path}")

## Evaluation

The same **Levenshtein distance** metric used during matching is also used here for evaluation, ensuring consistency between alignment and scoring.

Before computing distances, both transcription and stimulus are normalized to remove:

- punctuation
    
- case differences
    
- special tokens such as `<pause>`
    

This ensures that evaluation focuses on meaningful textual differences rather than formatting variations.

The average distance is computed only over successfully matched sentences, excluding `[not produced]` cases. This provides a clearer measure of transcription quality when a match exists.

In [24]:
def evaluate_results(matched):
    total = len(matched)
    not_produced = sum(1 for m in matched if m["transcription"] == "[not produced]")

    distances = []
    exact = 0

    for m in matched:
        if m["transcription"] == "[not produced]":
            continue

        dist = Levenshtein.distance(
            normalize(m["transcription"]),
            normalize(m["stimulus"])
        )

        distances.append(dist)

        if dist == 0:
            exact += 1

    avg_dist = sum(distances) / len(distances) if distances else 0

    print("\n─── EVALUATION ───")
    print(f"Total sentences: {total}")
    print(f"Matched: {total - not_produced}")
    print(f"Not produced: {not_produced}")
    print(f"Exact matches: {exact}")
    print(f"Average distance: {avg_dist:.2f}")

    return {
        "total": total,
        "matched": total - not_produced,
        "not_produced": not_produced,
        "exact": exact,
        "avg_distance": avg_dist
    }

In [27]:
def summarize_all_runs(all_metrics):
    print("\n---------------------- FINAL SUMMARY ----------------------\n")

    header = f"{'File':<8} | {'Matched':<8} | {'NotProd':<8} | {'Exact':<8} | {'AvgDist':<8}"
    print(header)
    print("-" * len(header))

    total_files = len(all_metrics)

    sum_matched = 0
    sum_not_produced = 0
    sum_exact = 0
    sum_avg_dist = 0

    for i, m in enumerate(all_metrics):
        file_name = f"File_{i+1}"

        print(f"{file_name:<8} | {m['matched']:<8} | {m['not_produced']:<8} | {m['exact']:<8} | {m['avg_distance']:<8.2f}")

        sum_matched += m["matched"]
        sum_not_produced += m["not_produced"]
        sum_exact += m["exact"]
        sum_avg_dist += m["avg_distance"]

    # Averages
    avg_matched = sum_matched / total_files
    avg_not_produced = sum_not_produced / total_files
    avg_exact = sum_exact / total_files
    avg_distance = sum_avg_dist / total_files

    print("-" * len(header))
    print(f"{'AVG':<8} | {avg_matched:<8.2f} | {avg_not_produced:<8.2f} | {avg_exact:<8.2f} | {avg_distance:<8.2f}")

## Pipeline

### Config

In [ ]:
TARGET_SR      = 16000

### End-to-End

In [26]:
def run_eit_pipeline(audio_path, stimuli, output_path,
                     target_sr=16000,
                     vad_thr=0.35,
                     pad_ms=200,
                     max_gap=0.5,
                     use_denoise=False):

    print(f"\n================ PROCESSING: {audio_path} ================\n")

    # LOAD
    audio, sr = load_and_resample(audio_path, target_sr)

    #  BASIC CLEANING
    audio = remove_dc_offset(audio)

    # VAD
    speech_segments = run_vad(audio, sr, thr=vad_thr)

    # CONTEXT PRESERVATION
    speech_segments = add_padding(speech_segments, sr, pad_ms=pad_ms)
    speech_segments = merge_vad_segments(speech_segments, max_gap=max_gap)

    # OPTIONAL DENOISE
    if use_denoise:
        audio = reduce_noise(audio, sr, pd=0.5)

    # TRANSCRIPTION
    results = transcribe(audio, sr, speech_segments)

    print("\n---- RAW SEGMENTS ----")
    for r in results:
        print(r["text"])

    # MERGE FRAGMENTED SENTENCES
    print("\n---- MERGING NEARBY FRAGMENTS ----")
    merged_results = merge_nearby_fragments(results, gap_threshold_s=2.0)

    print("\n---- MERGED SEGMENTS ----")
    for r in merged_results:
        print(r["text"])

    # MATCH TO STIMULI
    print("\n---- MATCHING (WINDOWING + CORRECTION) ----")
    matched = match_asr_to_stimuli(merged_results, stimuli)

    # EXPORT
    export_to_excel(matched, output_path)

    # EVALUATION
    metrics = evaluate_results(matched)

    return {
        "results": results,
        "merged_results": merged_results,
        "matched": matched,
        "metrics": metrics
    }

## RUN

### Running without noise reduction

In [28]:
if __name__ == "__main__":
    AUDIO_FILES = [
        "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3",
        "/content/drive/MyDrive/spanish/038011_EIT-1A.mp3",
        "/content/drive/MyDrive/spanish/038012_EIT-2A.mp3",
        "/content/drive/MyDrive/spanish/038015_EIT-1A.mp3",
    ]



    all_metrics = []

    for i, audio_path in enumerate(AUDIO_FILES):
        output_path = f"/content/drive/MyDrive/spanish/output_{i+1}.xlsx"

        result = run_eit_pipeline(audio_path, STIMULI, output_path)

        all_metrics.append(result["metrics"])


================ PROCESSING: /content/drive/MyDrive/spanish/038010_EIT-2A.mp3 ================



/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
[VAD] Found 46 speech segments


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  [73.9s → start + 2.0s] [es] nos fuimos al parque.
  [81.4s → start + 2.0s] [es] La llamaré mañana.
  [88.8s → start + 2.0s] [es] Puedes comprar carne en la tienda de butcher.
  [97.2s → start + 2.8s] [es] Mi hermano recibió un nuevo computador.
  [107.5s → start + 3.0s] [es] A veces toman a su papá para un paseo en el parque.
  [118.7s → start + 3.6s] [es] Vamos a jugar a la voleybal en el gimnasio que os he dicho.
  [165.1s → start + 2.0s] [es] quiero cortarme el pelo
  [172.4s → start + 2.0s] [es] El libro está en la mesa.
  [179.8s → start + 2.0s] [es] El caro lo tiene el pelo.
  [187.9s → start + 2.0s] [es] El se duche cada mañana.
  [197.0s → start + 2.8s] [es] ¿Qué dices ustedes que van a hacer hoy?
  [206.1s → start + 3.0s] [es] Duro que sepa manejar muy bien.
  [215.9s → start + 3.4s] [es] Las calles de esta ciudad son muy anches.
  [227.3s → start + 3.0s] [es] Puede que lleve mañana toda el día.
  [238.5s → start + 5.0s] [es] Las casas son muy bonitas, pero caras.
  [250.2s 

...

[VAD] Found 54 speech segments
  [59.6s → start + 2.0s] [es] nos fuimos al parque.
  [67.3s → start + 2.0s] [es] La llamaré mañana.
  [74.5s → start + 2.0s] [es] Puedes comprar carne en la tienda de butcher.
  [83.0s → start + 2.6s] [es] Mi hermano recibió un nuevo computador.
  [91.9s → start + 4.4s] [es] Sometimes they take their dog sometimes they take their dog for a walk in the park.
  [104.3s → start + 3.4s] [es] Vamos a jugar a la volea en el gimnasio que os he dicho.
  [150.8s → start + 2.0s] [es] Quiero cortarme a pelo.
  [158.0s → start + 2.0s] [es] El libro está en la mesa.
  [165.4s → start + 2.0s] [es] El carro no tiene pelo.
  [173.4s → start + 2.0s] [es] ¡Hasta la siguiente mañana!
  [182.6s → start + 2.4s] [es] ¿Qué dice usted en qué base oe?
  [191.6s → start + 3.0s] [es] tuve que manejar bien.
  [201.5s → start + 5.0s] [es] las caes en salida muchos años.
  [212.7s → start + 2.0s] [es] Puede que llevan mañana todos los días.
  [223.9s → start + 4.0s] [es] Las casas so

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 99 speech segments
  [1.4s → start + 3.0s] [es] At the end you're going to hit stop.
  [85.0s → start + 2.0s] [es] Te la llamaré mañana.
  [92.4s → start + 2.0s] [es] Puedes comprarme en la tienda de pusher.
  [101.2s → start + 2.0s] [es] Mi hermano recibió un nuevo computador.
  [111.5s → start + 2.6s] [es] A veces toman a su papá para caminar al parque.
  [123.0s → start + 3.0s] [es] Vamos a jugar volleyball en el gimnasio que te dije.
  [172.8s → start + 2.8s] [es] Quiero con Wernher Mero.
  [180.1s → start + 2.0s] [es] El peste es en la sala.
  [187.1s → start + 2.0s] [es] La tarea tiene la cara.
  [194.8s → start + 2.0s] [es] y dueme por la tarde
  [202.6s → start + 2.0s] [es] que deje cobrar a Ruy.
  [206.0s → start + 1.0s] [es] Que Dios goba.
  [211.8s → start + 2.0s] [es] Tú...
  [213.8s → start + 4.0s] [es] No sé, Kaya...
  [215.8s → start + 6.0s] [es] No sé...
  [222.5s → start + 4.0s] [es] las casas de ese país son más grandes.
  [234.1s → start + 3.0s] [es] Pued

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 41 speech segments
  [55.5s → start + 2.0s] [es] Y nos fuimos al parque.
  [64.8s → start + 2.0s] [es] La llamaré mañana.
  [72.1s → start + 2.0s] [es] Puedes comprarme en la tienda de butchers
  [80.8s → start + 2.6s] [es] Mi hermano recibió un nuevo computador.
  [90.9s → start + 3.0s] [es] A veces toman a su papá para una caminada en el parque.
  [101.9s → start + 3.0s] [es] Vamos a jugar volleyball en el gimnasio que os he dicho.
  [148.5s → start + 2.0s] [es] Quiero cortarme el pelo.
  [155.8s → start + 2.0s] [es] El libro está en la mesa.
  [163.1s → start + 2.0s] [es] El carro no tiene pelo.
  [171.3s → start + 2.0s] [es] El seducha cada mañana.
  [180.4s → start + 2.0s] [es] que dice a ustedes que vas a Arroyo.
  [189.2s → start + 2.0s] [es] ¿Tuvo que se permanecer bien?
  [199.2s → start + 3.8s] [es] Las calles de esta ciudad son muy anchas.
  [210.3s → start + 2.0s] [es] Puede que lleve mañana todo el día.
  [221.6s → start + 3.0s] [es] Las casas son muy bonitas, 

In [29]:
summarize_all_runs(all_metrics)


---------------------- FINAL SUMMARY ----------------------

File     | Matched  | NotProd  | Exact    | AvgDist 
----------------------------------------------------
File_1   | 24       | 6        | 5        | 6.12    
File_2   | 23       | 7        | 3        | 10.22   
File_3   | 14       | 16       | 0        | 11.43   
File_4   | 30       | 0        | 7        | 7.73    
----------------------------------------------------
AVG      | 22.75    | 7.25     | 3.75     | 8.88    


### Running with noise reduction

In [30]:
if __name__ == "__main__":
    AUDIO_FILES = [
        "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3",
        "/content/drive/MyDrive/spanish/038011_EIT-1A.mp3",
        "/content/drive/MyDrive/spanish/038012_EIT-2A.mp3",
        "/content/drive/MyDrive/spanish/038015_EIT-1A.mp3",
    ]



    all_metrics = []

    for i, audio_path in enumerate(AUDIO_FILES):
        output_path = f"/content/drive/MyDrive/spanish/output_{i+1}.xlsx"

        result = run_eit_pipeline(audio_path, STIMULI, output_path, use_denoise=True)

        all_metrics.append(result["metrics"])


================ PROCESSING: /content/drive/MyDrive/spanish/038010_EIT-2A.mp3 ================



Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 46 speech segments
[Noise Reduction] Done
  [73.9s → start + 2.0s] [es] nos fuimos al parque.
  [81.4s → start + 2.0s] [es] La llamaré mañana.
  [88.8s → start + 2.0s] [es] Puedes comprar carne en la tienda de butcher.
  [97.2s → start + 2.8s] [es] Mi hermano recibió un nuevo computador.
  [107.5s → start + 3.0s] [es] A veces toman a su papá para un paseo en el parque.
  [118.7s → start + 3.6s] [es] Vamos a jugar volleyball en el gimnasio que os he dicho.
  [165.1s → start + 2.0s] [es] Quiero cortarme el pelo.
  [172.4s → start + 2.0s] [es] El libro está en la mesa.
  [179.8s → start + 2.0s] [es] El caro lo tiene el pelo.
  [187.9s → start + 2.0s] [es] El se duche cada mañana.
  [197.0s → start + 2.8s] [es] ¿Qué dices ustedes que van a hacer hoy?
  [206.1s → start + 2.6s] [es] Duro que sepa manejar muy bien.
  [215.9s → start + 3.4s] [es] Las calles de esta ciudad son muy anches.
  [227.3s → start + 3.0s] [es] Puede que lleve mañana toda el día.
  [238.5s → start + 5.0s] [e

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 54 speech segments
[Noise Reduction] Done
  [59.6s → start + 2.0s] [es] nos fuimos al parque.
  [67.3s → start + 2.0s] [es] La llamaré mañana.
  [74.5s → start + 2.0s] [es] Puedes comprar carne en la tienda de butcher.
  [83.0s → start + 2.6s] [es] Mi hermano recibió un nuevo computador.
  [91.9s → start + 4.4s] [es] Sometimes they take their dog, sometimes they take their dog for a walk in the park.
  [104.3s → start + 3.4s] [es] Vamos a jugar a la volea en el gimnasio que os he dicho.
  [150.8s → start + 2.0s] [es] Quiero cortarme a pelo.
  [158.0s → start + 2.0s] [es] El libro está en la mesa.
  [165.4s → start + 2.0s] [es] El carro no tiene pelo.
  [173.4s → start + 2.0s] [es] ¡Hasta la siguiente mañana!
  [182.6s → start + 2.4s] [es] ¿Qué dice usted en qué base oe?
  [191.6s → start + 3.0s] [es] tuve que manejar bien.
  [201.5s → start + 5.0s] [es] las caes en salida muchos años.
  [212.7s → start + 2.0s] [es] Puede que llevan mañana todos los días.
  [223.9s → start +

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 99 speech segments
[Noise Reduction] Done
  [1.4s → start + 3.6s] [es] y haces tu recorrido. Al final, vas a decir STOP, ¿de acuerdo?
  [85.0s → start + 2.0s] [es] Te la llamaré mañana.
  [92.4s → start + 2.0s] [es] Puedes comprarme en la tienda de pusher.
  [101.2s → start + 2.0s] [es] Mi hermano recibió un nuevo computador.
  [111.5s → start + 2.6s] [es] A veces toman a su papá para caminar al parque.
  [123.0s → start + 3.0s] [es] Vamos a jugar volleyball en el gimnasio que te dije.
  [172.8s → start + 2.8s] [es] Quiero con R.V. Hermero.
  [180.1s → start + 2.0s] [es] El peste es en la sala.
  [187.1s → start + 2.0s] [es] La tarea tiene la cara.
  [194.8s → start + 2.0s] [es] y dueme por la tarde
  [202.6s → start + 2.0s] [es] que deje cobrar ahora.
  [206.0s → start + 1.0s] [es] Que Dios goba.
  [211.8s → start + 2.0s] [es] Tú...
  [213.8s → start + 4.0s] [es] No sé, Kaya...
  [215.8s → start + 6.0s] [es] No sé...
  [222.5s → start + 4.0s] [es] las casas de ese país son

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


[VAD] Found 41 speech segments
[Noise Reduction] Done
  [55.5s → start + 2.0s] [es] nos fuimos al parque.
  [64.8s → start + 2.0s] [es] La llamaré mañana.
  [72.1s → start + 2.0s] [es] Puedes comprarme en la tienda de butchers
  [80.8s → start + 2.6s] [es] Mi hermano recibió un nuevo computador.
  [90.9s → start + 3.0s] [es] A veces toman a su papá para caminar al parque.
  [101.9s → start + 3.0s] [es] Vamos a jugar volleyball en el gimnasio que os he dicho.
  [148.5s → start + 2.0s] [es] Quiero cortarme el pelo.
  [155.8s → start + 2.0s] [es] El libro está en la mesa.
  [163.1s → start + 2.0s] [es] El carro no tiene pelo.
  [171.3s → start + 2.0s] [es] El seducha cada mañana.
  [180.4s → start + 2.0s] [es] que dice a ustedes que vas a Arroyo.
  [189.2s → start + 2.0s] [es] ¿Tú lo que sepa manajar bien?
  [199.2s → start + 3.8s] [es] Las calles de esta ciudad son muy anchas.
  [210.3s → start + 2.0s] [es] Puede que lleve mañana todo el día.
  [221.6s → start + 3.0s] [es] Las casas son 

In [31]:
summarize_all_runs(all_metrics)


---------------------- FINAL SUMMARY ----------------------

File     | Matched  | NotProd  | Exact    | AvgDist 
----------------------------------------------------
File_1   | 24       | 6        | 5        | 6.12    
File_2   | 23       | 7        | 3        | 10.22   
File_3   | 14       | 16       | 0        | 11.64   
File_4   | 29       | 1        | 7        | 7.21    
----------------------------------------------------
AVG      | 22.50    | 7.50     | 3.75     | 8.80    
